# MyTravelHelper: AI-powered Travel Review Analyzer

**Student ID:** 24120056

**Project:** Building an intelligent travel review analysis system using Hugging Face models and Streamlit

This notebook presents a comprehensive overview of the MyTravelHelper application development process, including architecture, model selection, environment setup, and advanced NLP features for analyzing travel reviews.

## Table of Contents

1. [Objectives and Requirements Analysis](#objectives)
2. [Overall Architecture and Pipeline Design](#architecture)
3. [Environment Setup for Hugging Face](#setup)
4. [Model Introduction and Selection](#models)
5. [Basic Environment Testing](#testing)
6. [Application Demo and Usage](#demo)
7. [Advanced Feature 1: Aspect-Based Sentiment Analysis](#feature1)
8. [Advanced Feature 2: Topic Detection in Reviews](#feature2)
9. [Conclusions and Future Improvements](#conclusions)

## 1. Objectives and Requirements Analysis {#objectives}

### Project Objectives
Build an intelligent assistant to analyze travel reviews using Natural Language Processing (NLP) and provide actionable insights for travelers and hotel businesses.

### Functional Requirements
- **Sentiment Analysis:** Determine overall sentiment (positive/negative/neutral) of travel reviews
- **Aspect-Based Analysis:** Analyze sentiment for specific travel aspects (cleanliness, service, price, etc.)
- **Topic Detection:** Identify main topics/themes mentioned in reviews
- **User Interface:** Provide an intuitive web interface using Streamlit
- **Modular Architecture:** Separate concerns for models, utilities, and UI

### Non-Functional Requirements
- **Performance:** Fast inference (< 5 seconds per review)
- **Scalability:** Support batch processing if needed
- **Maintainability:** Well-documented, modular code
- **Reliability:** Graceful error handling

## 2. Overall Architecture and Pipeline Design {#architecture}

### System Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                     Streamlit Web Interface                      │
│  - Input: Travel Review Text                                    │
│  - Output: Analysis Results (Sentiment, Aspects, Topics)        │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│                    Text Preprocessing                            │
│  - Clean text (remove special chars, normalize spaces)          │
│  - Extract sentences and segments                               │
└────────────────────────┬────────────────────────────────────────┘
                         │
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
    ┌─────────────┐ ┌──────────┐ ┌──────────────┐
    │  Sentiment  │ │  Aspect  │ │    Topic     │
    │  Analysis   │ │ Sentiment│ │  Detection   │
    │(DistilBERT) │ │(Sentence │ │(Zero-shot)   │
    │             │ │  level)  │ │              │
    └─────────────┘ └──────────┘ └──────────────┘
          │              │              │
          └──────────────┼──────────────┘
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│                    Results Aggregation                           │
└─────────────────────────────────────────────────────────────────┘
```

### Data Flow Pipeline
1. **Input:** User enters travel review via Streamlit
2. **Preprocessing:** Text cleaned and normalized
3. **Analysis:** Multiple models process text in parallel
4. **Aggregation:** Results combined and formatted
5. **Visualization:** Results displayed with interactive UI

## 3. Environment Setup for Hugging Face {#setup}

### Prerequisites
- Python 3.8 or higher
- pip package manager
- Internet connection (for downloading models)
- At least 4GB free disk space

### Installation Steps

#### Step 1: Create Virtual Environment (Recommended)
```bash
# Windows
python -m venv myenv
myenv\\Scripts\\activate

# macOS/Linux
python3 -m venv myenv
source myenv/bin/activate
```

#### Step 2: Install Required Packages
```bash
pip install -r requirements.txt
```

#### Step 3: Verify Installation

In [ ]:
# Verify Python and package installation
import sys
print(f"Python version: {sys.version}")

import importlib
packages = ['streamlit', 'transformers', 'torch', 'numpy']
print("\\nInstalled packages:")
for package in packages:
    try:
        mod = importlib.import_module(package)
        version = getattr(mod, '__version__', 'version unknown')
        print(f"✓ {package}: {version}")
    except ImportError:
        print(f"✗ {package}: NOT INSTALLED")

## 4. Model Introduction and Selection {#models}

### Selected Models

#### 1. Sentiment Analysis Model
**Model:** `distilbert-base-uncased-finetuned-sst-2-english`
- **Architecture:** DistilBERT (distilled BERT)
- **Task:** Binary sentiment classification (Positive/Negative)
- **Training Data:** Stanford Sentiment Treebank (SST-2)
- **Advantages:**
  - 6x faster than BERT
  - 40% smaller than BERT
  - Retains 97% of BERT's performance
  - Perfect for real-time applications

#### 2. Zero-Shot Classification (Topic Detection)
**Model:** `facebook/bart-large-mnli`
- **Architecture:** BART (Bidirectional Autoregressive Transformer)
- **Task:** Zero-shot multi-class classification
- **Advantages:**
  - Classify to any topics without retraining
  - Flexible candidate label definition
  - Works well for topic extraction

### Model Comparison

| Feature | DistilBERT | BART-Large |
|---------|-----------|------------|
| Speed | Very Fast | Moderate |
| Accuracy | High | Very High |
| Model Size | 268MB | 1.6GB |
| Task | Sentiment | Classification |

### Model Loading

In [ ]:
from transformers import pipeline

print("Loading models...")
# Load sentiment analysis pipeline
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("✓ Sentiment analysis model loaded")

# Load zero-shot classification pipeline
topic_pipe = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
print("✓ Topic detection model loaded")
print("\\n✓ All models loaded successfully!")

## 5. Basic Environment Testing {#testing}

### Test 1: Sentiment Analysis

In [ ]:
# Test sentiment analysis
test_reviews = [
    "The hotel was absolutely amazing! Best stay ever!",
    "Terrible experience. Dirty rooms and rude staff.",
    "I love the beautiful views and friendly staff!"
]

print("SENTIMENT ANALYSIS TEST\\n" + "="*50)
for review in test_reviews:
    result = sentiment_pipe(review, truncation=True)
    label = result[0]["label"]
    score = result[0]["score"]
    icon = "😊" if label == "POSITIVE" else "😞"
    print(f"\\nReview: {review}")
    print(f"Result: {icon} {label} (score: {score:.4f})")

### Test 2: Topic Detection

In [ ]:
# Test topic detection
candidate_topics = [
    "cleanliness", "staff service", "location", 
    "food quality", "value for money"
]

sample_review = """
The hotel rooms were immaculate and the staff was incredibly helpful.
The location is perfect for visiting attractions. However, the food quality
was disappointing and the prices seemed a bit high.
"""

print("\\nTOPIC DETECTION TEST\\n" + "="*50)
print(f"Review: {sample_review.strip()}\\n")

result = topic_pipe(sample_review, candidate_topics, multi_class=True)
print("Detected Topics:")
for topic, score in zip(result["labels"], result["scores"]):
    print(f"  {topic:20} {score:.4f}")

### Test 3: Local Module Integration

In [ ]:
# Test local modules
import sys
sys.path.append('/d:/School/TDTT/24120056_12')

from src.utils import clean_text, analyze_text_statistics
from src.models import analyze_sentiment

print("\\nLOCAL MODULES TEST\\n" + "="*50)

# Test text cleaning
raw_text = "  The hotel   was   AMAZING!!!   Best experience ever...  "
cleaned = clean_text(raw_text)

print(f"\\nOriginal: '{raw_text}'")
print(f"Cleaned:  '{cleaned}'")

# Test statistics
test_review = "The hotel was great. The staff was friendly. The price was high."
stats = analyze_text_statistics(test_review)
print(f"\\nText Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 6. Application Demo and Usage {#demo}

### How to Run

#### Step 1: Install dependencies
```bash
pip install -r requirements.txt
```

#### Step 2: Test environment (optional)
```bash
python src/test_env.py
```

#### Step 3: Launch application
```bash
streamlit run src/streamlit_app.py
```

### Features
- **Overall Sentiment:** Primary sentiment and confidence score
- **Aspect-Based Analysis:** Sentiment for each aspect (cleanliness, staff, etc.)
- **Topic Detection:** Main topics mentioned in review
- **Text Statistics:** Word count, sentence count, etc.

## 7. Advanced Feature 1: Aspect-Based Sentiment Analysis {#feature1}

### Overview
Aspect-Based Sentiment Analysis (ABSA) identifies sentiment for specific aspects (cleanliness, staff, price) instead of just overall sentiment.

### Algorithm
```
For each aspect:
  1. Check if aspect is mentioned in review
  2. Extract sentences containing the aspect
  3. Analyze sentiment of each sentence
  4. Aggregate sentiment scores
  5. Return aspect-level sentiment
```

### Example

In [ ]:
import sys
sys.path.append('/d:/School/TDTT/24120056_12')

from src.models import analyze_aspect_sentiment
from src.utils import clean_text, extract_sentences_with_aspect

print("\\nADVANCED FEATURE 1: ASPECT-BASED SENTIMENT ANALYSIS\\n" + "="*60)

review = """
Excellent hotel with impeccable cleanliness! The staff was professional.
The location is perfect. However, prices were high and food was average.
"""

cleaned = clean_text(review)
print(f"Review: {cleaned}\\n")

aspects = ["cleanliness", "staff", "location", "price", "food"]
results = analyze_aspect_sentiment(cleaned, aspects)

print("Aspect-Based Analysis:\\n")
for aspect, result in results.items():
    if result["mentioned"]:
        emoji = "😊" if result["sentiment"] == "POSITIVE" else "😞"
        print(f"{emoji} {aspect.upper()}: {result['sentiment']} (score: {result['confidence_score']:.2%})")
    else:
        print(f"- {aspect.upper()}: Not mentioned")

### Use Cases
- Hotel management: identify aspects needing improvement
- Product development: understand feedback on specific features
- Marketing: highlight best aspects
- Training: focus training on poorly-reviewed aspects

## 8. Advanced Feature 2: Topic Detection in Reviews {#feature2}

### Overview
Topic detection uses zero-shot classification to identify main topics discussed in reviews without requiring labeled training data.

### How It Works
```
1. Input: Review text and candidate topic labels
2. Model converts to entailment problem
3. Compute entailment probabilities
4. Topics with highest scores = detected topics
5. Output: Ranked list of topics with scores
```

### Example

In [ ]:
import sys
sys.path.append('/d:/School/TDTT/24120056_12')

from src.models import detect_topics
from src.utils import clean_text

print("\\nADVANCED FEATURE 2: TOPIC DETECTION\\n" + "="*60)

reviews = [
    "Amazing location near beach! Perfect for vacation and sunsets.",
    "Flight delayed 3 hours. Poor airport service and transportation.",
    "Excellent restaurant with authentic food and great service."
]

topics = ["accommodation", "transportation", "food", "service", "location"]

for i, review in enumerate(reviews, 1):
    cleaned = clean_text(review)
    print(f"\\nReview {i}: {cleaned}")
    
    result = detect_topics(cleaned, topics)
    print("Topics (sorted by relevance):")
    for topic, score in zip(result["topics"], result["scores"]):
        if score > 0.1:
            print(f"  - {topic}: {score:.2%}")

### Applications
- Review categorization by topic
- Trend analysis in customer feedback
- FAQ generation
- Quality monitoring
- Competitive intelligence

## 9. Conclusions and Future Improvements {#conclusions}

### Project Summary
**MyTravelHelper** successfully demonstrates:
- ✓ Modular architecture with separation of concerns
- ✓ Integration of state-of-the-art Hugging Face models
- ✓ User-friendly Streamlit interface
- ✓ Comprehensive NLP pipeline
- ✓ Aspect-based sentiment analysis
- ✓ Advanced topic detection

### Key Achievements
1. Fast and accurate sentiment classification
2. Granular understanding of customer feedback
3. Flexible and dynamic topic identification
4. Production-ready, well-documented code
5. Intuitive user interface

### Future Improvements
- **Multi-language support** (French, Spanish, Vietnamese)
- **Batch processing** for multiple reviews
- **Data export** to CSV/JSON
- **Database integration** for historical analysis
- **API development** for system integration
- **Custom model fine-tuning** on domain-specific data
- **Advanced visualizations** and dashboards
- **Containerization** with Docker for deployment

### Technical Recommendations
- Use model caching to avoid reloading (✓ implemented)
- Implement input validation and sanitization
- Add comprehensive logging
- Consider async processing for scalability
- Monitor inference latency in production

## References

### Libraries
- **Transformers:** https://huggingface.co/transformers/
- **Streamlit:** https://streamlit.io/
- **PyTorch:** https://pytorch.org/

### Resources
- Hugging Face Model Hub: https://huggingface.co/models
- Streamlit Documentation: https://docs.streamlit.io/
- Transformers Documentation: https://huggingface.co/docs/transformers/

---

## Project Files Structure

```
24120056_12/
├── notebook.ipynb           # This notebook
├── requirements.txt         # Dependencies
├── src/
│   ├── __init__.py         # Package init
│   ├── models.py           # Model definitions
│   ├── utils.py            # Utility functions
│   ├── streamlit_app.py    # Main app
│   └── test_env.py         # Environment tests
└── README.md               # Documentation
```

**End of Notebook**

# MyTravelHelper: AI-powered Travel Review Analyzer

This notebook presents the process of building the MyTravelHelper application using Streamlit for the interface. The application leverages Hugging Face models for sentiment analysis, intent classification, and topic detection in travel reviews.

## Table of Contents
1. Objectives and Requirements Analysis
2. Overall Architecture (Pipeline Diagram)
3. Environment Setup for Hugging Face Inference Providers
4. Model Introduction
5. Basic Environment Test
6. Running and Testing the Application
7. Advanced Features

## 1. Objectives and Requirements Analysis
- **Objective:** Build an AI-powered assistant to analyze travel reviews and user queries.
- **Requirements:**
  - Sentiment analysis (aspect-based) for travel reviews.
  - Intent classification and extraction of user requirements.
  - Topic detection in travel reviews.
  - User-friendly interface using Streamlit.
  - Modular, maintainable codebase.

## 2. Overall Architecture (Pipeline Diagram)

```mermaid
graph TD
    A[User Input] --> B[Preprocessing] --> C[Sentiment Analysis] --> D[Intent Classification] --> E[Topic Detection] --> F[Results Display] 
```

- **User Input:** User enters a travel review or query.
- **Preprocessing:** Clean and prepare text.
- **Sentiment Analysis:** Analyze sentiment (aspect-based).
- **Intent Classification:** Detect user intent.
- **Topic Detection:** Identify main topics/aspects.
- **Results Display:** Show results in Streamlit UI.

## 3. Environment Setup for Hugging Face Inference Providers

Install required packages (run in terminal):

```bash
pip install streamlit transformers
```

- Ensure you have Python 3.8+ installed.
- Internet connection is required for downloading models.

## 4. Model Introduction

- **Sentiment Analysis:** `cardiffnlp/twitter-roberta-base-sentiment-latest` (or default).
- **Intent Classification:** `mrm8488/bert-tiny-finetuned-sms-spam-detection` (or similar).
- **Topic Detection:** `facebook/bart-large-mnli` (zero-shot classification).

All models are loaded via Hugging Face's `transformers.pipeline`.

In [4]:
# Example: Load sentiment analysis pipeline
from transformers import pipeline
sentiment_pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
sentiment_pipe("The hotel was clean and the staff were friendly.")

NameError: name 'torch' is not defined

## 5. Basic Environment Test

Test if Hugging Face pipelines work correctly.

In [ ]:
# Run the environment test script
!python src/test_env.py

## 6. Running and Testing the Application

To start the Streamlit app, run in terminal:

```bash
streamlit run src/streamlit_app.py
```

- Enter a travel review or query in the UI.
- The app will display sentiment, intent, topics, and aspects.

## 7. Advanced Features

### 7.1 Sentiment Analysis for Travel Reviews (Aspect-based)
- The app uses a sentiment analysis pipeline and extracts aspects (e.g., food, location, service) from the review.
- Results are shown for each aspect mentioned.

### 7.2 Intent Classification and Extraction
- The app classifies the user's intent (e.g., asking for recommendations, reporting an issue).
- Extracts user requirements from the input.

### 7.3 Topic Detection in Travel Reviews
- Uses zero-shot classification to detect topics/aspects in the review.
- Shows the most relevant topics.

In [ ]:
# Example: Use all pipelines together
from src.models import get_sentiment_pipeline, get_intent_pipeline, get_topic_pipeline
from src.utils import clean_text, extract_aspects

review = "The hotel was clean, but the food was disappointing and the location was perfect."
cleaned = clean_text(review)
sentiment = get_sentiment_pipeline()(cleaned)
intent = get_intent_pipeline()(cleaned)
topics = get_topic_pipeline()(cleaned, ["food", "location", "service"])
aspects = extract_aspects(cleaned, ["food", "location", "service"])
print("Sentiment:", sentiment)
print("Intent:", intent)
print("Topics:", topics)
print("Aspects:", aspects)